In [ ]:
from heat_problem    import HeatProblem
from hngd_problem import HNGDProblem

# --- Heat solve ---
heat = HeatProblem(mesh, k=18.0, Q=Q_from_openmc)

T_pipe = fem.Constant(mesh, PETSc.ScalarType(1000.0))
dofs   = fem.locate_dofs_topological(heat.V, fdim, heat_pipe_facets)
bc     = fem.dirichletbc(T_pipe, dofs, heat.V)

heat.setup(bcs=[bc], radiative_bcs=[{
    'facet_tags': facet_tags,
    'tag'       : fuel_pin_tag,
    'T_source'  : 1025.0,
    'emissivity': 0.8,
}])
heat.solve()
heat.export('results/temperature.bp')

# --- Hydrogen transport (steady state, Fick + Soret only) ---
mat = HNGDMaterial(D0=1.53e-7, E_D=0.145,
                   KD0=0.0, KN0=0.0, Kth0=0.0, Kmob0=0.0)
source = HNGDSource(mat, heat.T, KG_override=0.0)

hngd = HNGDProblem(mesh, source, initial_conditions={'C_SS': 150.0, 'C_prec': 0.0})
hngd.setup(bcs=[], include_soret=True, steady_state=True)
hngd.run(t_final=1.0, dt=1.0, export_bp='results/hydrogen.bp')